In [0]:
# Define widgets with default values
dbutils.widgets.text("quote_id", "PR9999")
dbutils.widgets.text("catalog", "lrcatalog")
dbutils.widgets.text("schema", "agentic_underwriting")

#%pip install -U databricks-agents databricks-openai databricks-langchain mlflow
#dbutils.library.restartPython()


In [0]:
# Get widget values
quote_id = dbutils.widgets.get("quote_id")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

### Here we start with tool definitions for our agent.


### 🔍 Function: `make_get_quote_details`

This function factory returns a callable that retrieves details of a specific insurance quote from the `quotes` table in the specified catalog and schema.

- **Input:** `quote_id` (case-insensitive, trimmed string)
- **Output:** Markdown-formatted table with quote details, or an error message if not found
- **Use case:** Can be plugged into LangChain agents or UI apps to display quote details based on user input

In [0]:
def make_get_quote_details(catalog, schema):
    def _fn(quote_id: str) -> str:
        quote_id_clean = quote_id.strip()
        df = spark.sql(f"""
            SELECT * FROM {catalog}.{schema}.property_quotes
            WHERE LOWER(quote_id) = LOWER('{quote_id_clean}')
        """).toPandas()

        if df.empty:
            return f"🚫 Quote not found for quote_id: {quote_id_clean}"
        
        markdown = df.to_markdown(index=False)
        return f"✅ Quote found:\n\n{markdown}"
    return _fn


### 🔍 Function: `make_validate_claims`

This tool will:
	•	Take a quote_id
	•	Look up the customer in {catalog}.{schema}.property_quotes
	•	Find the corresponding record in {catalog}.{schema}.global_claims_disclosure_validation (match on name + postcode)
	•	Return whether declared vs. actual claims match, along with details

In [0]:
def make_validate_claims(catalog, schema):
    def _fn(quote_id: str) -> str:
        quote_id_clean = quote_id.strip()

        # Get quote details first to match on customer/postcode
        quote_df = spark.sql(f"""
            SELECT first_name, last_name, postcode, previous_claims
            FROM {catalog}.{schema}.property_quotes
            WHERE LOWER(quote_id) = LOWER('{quote_id_clean}')
        """).toPandas()

        if quote_df.empty:
            return f"🚫 Quote not found for quote_id: {quote_id_clean}"

        first_name = quote_df.iloc[0]["first_name"]
        last_name = quote_df.iloc[0]["last_name"]
        postcode = quote_df.iloc[0]["postcode"]
        declared_claims = quote_df.iloc[0]["previous_claims"]

        # Pull from claims disclosure validation table
        claims_df = spark.sql(f"""
            SELECT first_name, last_name, postcode, claims_amount
            FROM {catalog}.{schema}.global_claims_disclosure_validation
            WHERE LOWER(first_name) = LOWER('{first_name}')
              AND LOWER(last_name) = LOWER('{last_name}')
              AND postcode = '{postcode}'
        """).toPandas()

        if claims_df.empty:
            return f"⚠️ No claims disclosure record found for {first_name} {last_name}, {postcode}."

        actual_claims = claims_df.iloc[0]["claims_amount"]

        # Compare declared vs. actual
        if declared_claims == actual_claims:
            status = "✅ Claims match"
        else:
            status = f"❌ Mismatch: declared {declared_claims}, actual {actual_claims}"

        markdown = claims_df.to_markdown(index=False)
        return f"{status}\n\nDisclosure record:\n\n{markdown}"
    return _fn


### 🔍 Function: `make_get_call_transcript`

This function factory returns a callable that retrieves the call transcript associated with a specific motor insurance quote from the `sales_call_transcripts` table.

- **Input:** `quote_id` (string)
- **Output:** Raw text of the call transcript, or a message if no transcript is found
- **Use case:** Can be used in LangChain agents or apps to provide customer interaction history for underwriting or sales analysis

In [0]:
def make_get_call_transcript(catalog, schema):
    def _fn(quote_id: str) -> str:
        quote_id_clean = quote_id.strip()

        # Try direct lookup by quote_id
        df = spark.sql(f"""
            SELECT quote_id, call_transcript
            FROM {catalog}.{schema}.global_sales_call_transcripts
            WHERE LOWER(quote_id) = LOWER('{quote_id_clean}')
        """).toPandas()

        # Fallback: search by postcode from property_quotes if no direct match
        if df.empty:
            qdf = spark.sql(f"""
                SELECT first_name, last_name, postcode
                FROM {catalog}.{schema}.property_quotes
                WHERE LOWER(quote_id) = LOWER('{quote_id_clean}')
            """).toPandas()

            if qdf.empty:
                return f"🚫 No transcript found and quote not found for fallback. quote_id: {quote_id_clean}"

            postcode = qdf.iloc[0]["postcode"]

            df = spark.sql(f"""
                SELECT quote_id, call_transcript
                FROM {catalog}.{schema}.global_sales_call_transcripts
                WHERE call_transcript LIKE '%{postcode}%'
            """).toPandas()

            if df.empty:
                return f"⚠️ No transcript found for quote_id {quote_id_clean}. Searched by postcode {postcode} as fallback."

        # Format transcripts (truncate very long ones)
        parts = []
        for _, row in df.iterrows():
            text = row["call_transcript"] or ""
            if len(text) > 2000:
                text = text[:2000] + "... [truncated]"
            parts.append(f"### Transcript for `{row['quote_id']}`\n\n```\n{text}\n```")

        return "📞 Call transcript(s) found:\n\n" + "\n\n---\n\n".join(parts)

    return _fn


### 🔍 Function: `score_quote_tool`

This function scores a motor insurance quote using a basic risk model based on age, vehicle type, number of claims, and no claims discount (NCD).

- **Input:** A dictionary or JSON string with keys: `age`, `vehicle_type`, `ncd_declared`, and `claims_declared`
- **Output:** A dictionary containing the calculated `price`, the `model_name`, and optionally an `error` message
- **Use case:** Can be used in LangChain agents or quote evaluation tools to simulate a pricing model based on risk factors

In [0]:
from pyspark.sql.functions import col, when, round as round_col, coalesce, lit

def make_score_property_quote(catalog, schema):
    def _fn(quote_id: str) -> str:
        qid = quote_id.strip()

        # Load the target quote
        qdf = spark.sql(f"""
            SELECT *
            FROM {catalog}.{schema}.property_quotes
            WHERE LOWER(quote_id) = LOWER('{qid}')
        """)
        if qdf.limit(1).count() == 0:
            return f"🚫 Quote not found for quote_id: {qid}"

        # Optionally join postcode-level attributes
        attrs_fqtn = f"{catalog}.{schema}.property_attributes"
        if spark.catalog.tableExists(attrs_fqtn):
            adf = spark.table(attrs_fqtn).select("postcode","property_risk_level","has_garage","has_driveway")
            quotes_df = qdf.join(adf, on="postcode", how="left")
        else:
            quotes_df = (qdf
                .withColumn("property_risk_level", lit(None).cast("string"))
                .withColumn("has_garage", lit(False))
                .withColumn("has_driveway", lit(False))
            )

        # --------------------------
        # Multipliers & adjustments
        # --------------------------

        # Property type multiplier
        prop_type_mult = (
            when(col("property_type") == "Detached",      1.10)
            .when(col("property_type") == "Semi-Detached",1.05)
            .when(col("property_type") == "Terraced",     1.00)
            .when(col("property_type") == "Flat",         0.95)
            .when(col("property_type") == "Bungalow",     1.00)
            .otherwise(1.00)
        )

        # Risk band multiplier (from attributes)
        risk_mult = (
            when(col("property_risk_level") == "high",     1.30)
            .when(col("property_risk_level") == "mid_high",1.15)
            .when(col("property_risk_level") == "mid_low", 1.05)
            .when(col("property_risk_level") == "low",     1.00)
            .otherwise(1.00)
        )

        # Construction adjustment
        construction_adj = (
            when(col("construction_type") == "Timber",   100)
            .when(col("construction_type") == "Concrete", 50)
            .when(col("construction_type") == "Stone",    20)
            .when(col("construction_type") == "Brick",     0)
            .otherwise(25)
        )

        # Roof adjustment
        roof_adj = (
            when(col("roof_type") == "Thatched", 300)
            .when(col("roof_type") == "Flat Roof", 100)
            .when(col("roof_type") == "Slate", 20)
            .when(col("roof_type") == "Tile", 0)
            .otherwise(25)
        )

        # Occupancy adjustment
        occupancy_adj = (
            when(col("occupancy_status") == "Vacant",         200)
            .when(col("occupancy_status") == "Tenant",        100)
            .when(col("occupancy_status") == "Owner-Occupied",  0)
            .otherwise(50)
        )

        # Bedrooms adjustment (centered at 3, £20 per bedroom step)
        bedrooms_adj = (coalesce(col("num_bedrooms"), lit(3)) - lit(3)) * lit(20)

        # Property age adjustment
        yb = coalesce(col("year_built"), lit(1995))
        age_adj = (
            when(yb < 1950, 150)
            .when(yb < 1975, 100)
            .when(yb < 2000,  50)
            .otherwise(0)
        )

        # Previous claims adjustment
        claims_adj = coalesce(col("previous_claims"), lit(0)) * lit(150)

        # Protection discounts
        garage_disc   = when(col("has_garage") == True,   -30).otherwise(0)
        driveway_disc = when(col("has_driveway") == True, -10).otherwise(0)

        # Channel tweak
        channel_adj = when(col("channel") == "aggregator", 15).otherwise(0)

        # Base and final price
        base_price_expr = (
            lit(300) + construction_adj + roof_adj + occupancy_adj +
            bedrooms_adj + age_adj + claims_adj + garage_disc + driveway_disc + channel_adj
        )

        scored = (quotes_df
            .withColumn("prop_type_mult", prop_type_mult)
            .withColumn("risk_mult", risk_mult)
            .withColumn("base_price", round_col(base_price_expr, 2))
            .withColumn("quote_value", round_col(base_price_expr * prop_type_mult * risk_mult, 2))
        )

        # Pull single row and present a compact breakdown
        pdf = scored.toPandas()
        if pdf.empty:
            return f"🚫 Unable to score quote_id: {qid}"

        row = pdf.iloc[0]

        # Build a readable summary
        lines = [
            f"### 🧮 Property Quote Score for `{qid}`",
            "",
            f"- **Property type**: {row.get('property_type')}",
            f"- **Construction / Roof**: {row.get('construction_type')} / {row.get('roof_type')}",
            f"- **Occupancy**: {row.get('occupancy_status')}",
            f"- **Bedrooms / Year built**: {row.get('num_bedrooms')} / {row.get('year_built')}",
            f"- **Previous claims**: {row.get('previous_claims')}",
            f"- **Postcode**: {row.get('postcode')}",
            f"- **Attrs**: risk={row.get('property_risk_level')}, garage={row.get('has_garage')}, driveway={row.get('has_driveway')}",
            "",
            f"**Base price**: £{row.get('base_price')}",
            f"**Multipliers**: prop_type={row.get('prop_type_mult')}, risk={row.get('risk_mult')}",
            f"**Final quote_value**: **£{row.get('quote_value')}**",
        ]

        return "\n".join(lines)

    return _fn


### 🏡 Function: `validate_property_attributes`

This function retrieves property-level risk data for a given postcode, such as garage and driveway availability and overall property risk level.

- **Input:** A UK postcode as a plain string (e.g., `'CR3 6JE'`)
- **Output:** A markdown table of the matching row from the `property_attributes` table, or a message if no match is found
- **Use case:** Used in LangChain agents or underwriting pipelines to enrich or validate property context based on location

In [0]:
def make_validate_property_risk(catalog, schema):
    def _fn(quote_id: str) -> str:
        qid = quote_id.strip()

        # Get postcode for this quote
        qdf = spark.sql(f"""
            SELECT postcode
            FROM {catalog}.{schema}.property_quotes
            WHERE LOWER(quote_id) = LOWER('{qid}')
        """).toPandas()

        if qdf.empty:
            return f"🚫 Quote not found for quote_id: {qid}"

        postcode = qdf.iloc[0]["postcode"]

        # Look up property attributes
        attrs_df = spark.sql(f"""
            SELECT *
            FROM {catalog}.{schema}.property_attributes
            WHERE postcode = '{postcode}'
        """).toPandas()

        if attrs_df.empty:
            return f"⚠️ No property_attributes record found for postcode {postcode} (quote {qid})."

        row = attrs_df.iloc[0]

        # Validation checks
        issues = []
        if row.get("property_risk_level") not in ["low","mid_low","mid_high","high"]:
            issues.append(f"Invalid risk level: {row.get('property_risk_level')}")
        if pd.isna(row.get("has_garage")):
            issues.append("Missing value: has_garage")
        if pd.isna(row.get("has_driveway")):
            issues.append("Missing value: has_driveway")

        # Build response
        if not issues:
            status = "✅ Property risk features validated successfully"
        else:
            status = "❌ Validation issues:\n- " + "\n- ".join(issues)

        markdown = attrs_df.to_markdown(index=False)
        return f"{status}\n\nRecord for postcode `{postcode}`:\n\n{markdown}"

    return _fn


### 🧰 Tool Definitions for LangChain Agent

This section defines the set of tools available to the LangChain agent, each wrapping a callable function with a clear name and description.

Tools included:
	•	Get Quote Details: Returns declared customer and quote info by quote ID.
	•	Validate Claims: Compares declared claims against disclosure records using name and postcode, and flags discrepancies.
	•	Get Call Transcript: Retrieves the sales call transcript for a given quote ID, providing conversational context around the quote.
	•	Score Quote: Calculates a property insurance premium using risk-based adjustments and multipliers (property type, construction, roof, occupancy, claims, postcode features).
	•	Validate Property Risk Features: Cross-checks property-level attributes (risk band, year built, bedrooms, garage/driveway) and flags missing or inconsistent values.

Use case:
These tools enable the agent to gather and validate underwriting data across customer declarations, third-party checks, and sales interactions. They provide a consistent way to simulate pricing outcomes, highlight data quality issues, and prepare structured insights for the human underwriter.

In [0]:
from langchain.tools import Tool

tools = [
    Tool.from_function(
        make_get_quote_details(catalog, schema),
        name="Get Quote Details",
        description=(
            "Returns the declared property quote data including first name, last name, postcode, "
            "age, property type, construction type, roof type, number of bedrooms, year built, "
            "occupancy status, declared previous claims and channel. "
            "Input: just the quote ID value (e.g., 'PR9999')."
        )
    ),
    Tool.from_function(
        make_validate_claims(catalog, schema),
        name="Validate Claims",
        description=(
            "Checks declared previous claims in the property quote against the verified claims "
            "from global_claims_disclosure_validation using first name, last name and postcode. "
            "Returns whether declared and actual claims match or differ. "
            "Input: just the quote ID value."
        )
    ),
    Tool.from_function(
        make_get_call_transcript(catalog, schema),
        name="Get Call Transcript",
        description=(
            "Returns the sales call transcript text for a given quote ID if available. "
            "Analyse the transcript and compare with the property quote and validation data. "
            "Useful to check for misstatements or discrepancies made during the sales call. "
            "Input: just the quote ID value."
        )
    ),
    Tool.from_function(
        make_score_property_quote(catalog, schema),
        name="Score Property Quote",
        description=(
            "Applies the property pricing logic to calculate a premium (quote_value) for a given quote. "
            "Considers property type, construction, roof, occupancy, bedrooms, year built, previous claims, "
            "channel, and postcode-level risk features. "
            "Input: just the quote ID value. Returns a breakdown of adjustments and the final price."
        )
    ),
    Tool.from_function(
        make_validate_property_risk(catalog, schema),
        name="Validate Property Attributes",
        description=(
            "Validates property risk features for the postcode linked to a given quote ID. "
            "Checks attributes such as has_garage, has_driveway, and property_risk_level. "
            "Flags missing or invalid values. "
            "Input: just the quote ID value."
        )
    )
]


### 🧠 Agent Initialization

This cell initializes a LangChain agent using a Databricks-hosted LLM (`meta-llama-3-1-70b-instruct`) and a set of predefined tools.

- **LLM:** `ChatDatabricks` connected to a specified endpoint
- **Agent type:** `ZERO_SHOT_REACT_DESCRIPTION` — allows reasoning over tool descriptions without examples
- **Settings:**
  - `verbose=True` for step-by-step logging
  - `handle_parsing_errors=True` to gracefully manage response formatting issues

- **Use case:** Powers a dynamic, tool-using agent capable of answering insurance-related questions or performing quote evaluations

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.agents import initialize_agent, AgentType

llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)


### 📝 Agent Execution Workflow

This cell runs a multi-step underwriting evaluation process through the LangChain agent using the specified quote ID.

- **Goal:** Assist an underwriter in checking quote accuracy and flagging issues before approval
- **Steps:**
  1. **Get Quote Details** – fetch declared quote information
  2. **Validate Claims** – check against verified claims/NCD data
  3. **Get Call Transcript** – verify customer disclosures against quote data
  4. **Identify Errors** – if call center input error is suspected, rescore using `Score Quote`
  5. **Summarize** – return a comprehensive decision message

- **Use case:** Enables end-to-end decision support with data validation, transcript analysis, and dynamic pricing review

In [0]:
agent_output = agent.run(f"""
You are an underwriter for property insurance policies. You will check a quote and decide whether it should be approved or reviewed.

Step 1: Use the 'Get Quote Details' tool with quote ID {quote_id}. If the quote is not found, respond with a message and stop the process. Otherwise, get the quote details and return the message.

Step 2: Use the 'Validate Claims' tool. If the customer does not exist in validation data, flag potential fraud, respond with the correct message, and stop this process. Otherwise, check the number of claims, compare to the quote, and decide if it should be approved or reviewed. Return the relevant message.

Step 3: Use the 'Get Call Transcript' tool to retrieve the sales call for this quote. Check if the values mentioned by the customer in the call (e.g. postcode, property type, occupancy, claims) match what is shown in the quote. If the customer provided correct details in the call but the quote shows different values, this may indicate an error made by the call handler.

Step 4: Use the 'Validate Property Attributes' tool to retrieve the property details for this postcode. Check if the returned attributes (e.g. has_garage, has_driveway, property_risk_level) align with what is assumed or declared in the quote. If the quote relies on incorrect property assumptions—for example, assuming a garage where none exists—this could indicate an underwriting error or misclassification. Flag the mismatch and suggest review, then run Step 5 for a new quote with corrected parameters. Ensure to include in the description the results of the search in comparison to the quote data. If there is no mismatch, skip Step 5 and go to Step 6.

Step 5: Don't use this if the data matches in previous steps or if there is no call transcript. If a mismatch was likely caused by incorrect data entry during the call, note this and suggest review. Use the 'Score Property Quote' tool to create a new price with corrected values. Otherwise, list all mismatches clearly. Return a full description of this process.

Step 6: Summarize the above steps in points with details on what tools were used and their results. Ensure to include all details regarding data you have gathered during previous steps. Say if the quote should be approved or rejected. Add a dad joke at the end.
""")


### 💾 Save Agent Output to Table

This cell saves the agent's underwriting decision into the `agent_review` table in Unity Catalog.

- **Steps:**
  1. Creates a single-row DataFrame with `quote_id` and `agent_output`
  2. Registers it as a temporary view (`new_review_data`)
  3. Executes a `MERGE` to upsert the result into `lrcatalog.agentic_underwriting.agent_review`

- **Use case:** Stores decisions and justifications made by the agent for audit, governance, or follow-up review

In [0]:
#save output into a table
from pyspark.sql import Row

# Create a Spark DataFrame with new output
row = Row(quote_id=quote_id, agent_output=agent_output)
new_data = spark.createDataFrame([row])

# Register as temp view for merge
new_data.createOrReplaceTempView("new_review_data")

# Run MERGE to update or insert
spark.sql(f"""
MERGE INTO {catalog}.{schema}.global_agent_output AS target
USING new_review_data AS source
ON target.quote_id = source.quote_id
WHEN MATCHED THEN UPDATE SET target.agent_output = source.agent_output
WHEN NOT MATCHED THEN INSERT (quote_id, agent_output) VALUES (source.quote_id, source.agent_output)
""")

In [0]:
import json

result_summary = {
    "status": "success",
    "quote_id": quote_id,
    "catalog": catalog,
    "schema": schema,
    "table": "underwriting_results"
}

dbutils.notebook.exit(json.dumps(result_summary))